In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
print("All libraries imported successfully")

All libraries imported successfully


In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of transactions to generate
n = 1000

# Build the DataFrame column by column
df = pd.DataFrame({
    # Random amounts - exponential dist mimics real spending
    "amount": np.random.exponential(scale=500, size=n),
    
    # Randomly credit or debit
    "type": np.random.choice(["credit", "debit"], size=n),
    
    # Country with realistic probabilities
    "country": np.random.choice(
        ["US", "UK", "IN", "NG", "RU"],
        size=n,
        p=[0.5, 0.2, 0.15, 0.1, 0.05]
    ),
    
    # Customer age between 18 and 70
    "customer_age": np.random.randint(18, 70, size=n),
    
    # Hour of transaction 0-23
    "hour": np.random.randint(0, 24, size=n)
})

# Start everyone as legitimate (0)
df["fraud"] = 0

# Flag high amount from high risk countries as fraud
df.loc[(df["amount"] > 1500) & 
       (df["country"].isin(["NG", "RU"])), "fraud"] = 1

# Flag high amount at unusual hours as fraud
df.loc[(df["amount"] > 2000) & 
       (df["hour"] < 6), "fraud"] = 1

# Check shape and fraud distribution
print(df.shape)
print(df["fraud"].value_counts())
print(df.head())

(1000, 6)
fraud
0    988
1     12
Name: count, dtype: int64
        amount    type country  customer_age  hour  fraud
0   234.634045  credit      UK            68    12      0
1  1505.060715   debit      US            64     7      0
2   658.372847   debit      US            64     6      0
3   456.471277   debit      US            21    16      0
4    84.812435   debit      US            24    20      0


#### Feature Preparation

In [4]:
# Step 1 - Make a copy so original data is preserved
df_model = df.copy()

# Step 2 - Encode text columns to numbers
le = LabelEncoder()

# credit/debit → 1/0
df_model["type"] = le.fit_transform(df_model["type"])

# US/UK/IN/NG/RU → 4/3/0/1/2 (alphabetical order)
df_model["country"] = le.fit_transform(df_model["country"])

# Step 3 - Separate features (X) and label (y)
# X = what the model uses to make predictions
# y = what we want the model to predict

X = df_model[["amount", "type", "country", "customer_age", "hour"]]
y = df_model["fraud"]

# Step 4 - Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
   X, y,
    test_size= 0.2,
    random_state= 42,
    stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nFraud cases in training:", y_train.sum())
print("Fraud cases in testing:", y_test.sum())

Training set size: (800, 5)
Testing set size: (200, 5)

Fraud cases in training: 10
Fraud cases in testing: 2


#### Train and Evaluate

In [5]:
# Step 1 - Create the model
model = LogisticRegression(random_state=42, max_iter=1000)

# Step 2 - Train the model
model.fit(X_train, y_train)

# Step 3 - Make predictions on test set
y_pred = model.predict(X_test)

# Step 4 - Evaluate performance
print("===confusion matrix===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

===confusion matrix===
[[196   2]
 [  1   1]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       198
           1       0.33      0.50      0.40         2

    accuracy                           0.98       200
   macro avg       0.66      0.74      0.70       200
weighted avg       0.99      0.98      0.99       200



Precision 0.33 for fraud:

Of all transactions flagged as fraud, only 33% were actually fraud. 2 flags raised, only 1 was real fraud.
Recall 0.50 for fraud:

Of all actual frauds, we caught 50%. Missed 1 out of 2 fraud cases.
Accuracy 0.98:

Looks impressive but misleading — model is mostly just predicting everything as legitimate.

##### Add Class Weights

In [6]:
# Tell model to penalize missing fraud more heavily
model_weighted = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced" # automatically handles imbalance
)

model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print("===confusion matrix===")
print(confusion_matrix(y_test, y_pred_weighted))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_weighted))

===confusion matrix===
[[192   6]
 [  0   2]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       198
           1       0.25      1.00      0.40         2

    accuracy                           0.97       200
   macro avg       0.62      0.98      0.69       200
weighted avg       0.99      0.97      0.98       200



#### We can conclude that model 2 is better than model 1